# Data Preparation - Olist Marketplace
Daten strukturiert und reproduzierbar verarbeiten

## Verbindung mit Duckdb

In [ ]:
import duckdb
from pathlib import Path
import pandas as pd

# Projektpfade
RAW_PATH = Path("../data/raw/brazilian-ecommerce")
DB_PATH = Path("../data/processed/olist.duckdb")

# Verbindung zur DuckDB als Datei (persistente DB statt in-memory)
con = duckdb.connect(DB_PATH.as_posix())

# Hilfsfunktion für SQL-Abfragen
def sql(query: str):
    return con.execute(query).df()

# Alle CSV-Dateien in DuckDB als Tabellen speichern (persistent in olist.duckdb)
for file in RAW_PATH.glob("*.csv"):
    table_name = file.stem.replace("olist_", "").replace("_dataset", "")

    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{file.as_posix()}');
    """)

# Alle Tabellen anzeigen
sql("SHOW TABLES")

## Erstellung der Dataframes pro Kernaufgabe

### EDA für erste Kernaufgabe

In [ ]:
# Dataframe für Aufgabe 1. für EDA
df_rfm_eda = sql("""
SELECT 
    c.customer_id,
    c.customer_unique_id,
    c.customer_city,
    c.customer_state,
    c.customer_zip_code_prefix,
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    oi.price,
    oi.freight_value,
    Count(oi.product_id) AS product_count,
    
FROM customers AS c
JOIN orders o 
        ON c.customer_id = o.customer_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
GROUP BY c.customer_id, o.order_id, o.order_purchase_timestamp, 
                 o.order_approved_at, oi.price, oi.freight_value, 
                 c.customer_unique_id, c.customer_city, c.customer_state, 
                 c.customer_zip_code_prefix, o.order_status
    """)

In [ ]:
df_rfm_eda

In [ ]:
df_rfm_eda.describe()

#### Auffälligkeiten

1. Bei order_purchase_timestamp und order_approved_at scheint es NaN werte zu geben
2. 

In [ ]:
df_rfm_eda.dtypes

In [ ]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = {'customer_id': 'category', 
              'customer_unique_id': 'category', 
              'customer_city': 'category', 
              'customer_state': 'category', 
              'customer_zip_code_prefix': 'category', 
              'order_id': 'category', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'price': 'float32',
              'freight_value': 'float32',
              'product_count': 'int16', }
df_rfm_eda = df_rfm_eda.astype(col_dtypes)
df_rfm_eda.dtypes

In [ ]:
df_rfm_eda.describe()

In [ ]:
df_rfm_eda.isna().sum()

In [ ]:
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

In [ ]:
pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_approved_at'].isna())

In [ ]:
df_rfm_eda['order_approved_at'] = df_rfm_eda['order_approved_at'].fillna(
    pd.to_datetime(df_rfm_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_rfm_eda[df_rfm_eda.isna().any(axis=1)].head(15)

In [ ]:
print("Gesamte Duplikate:", df_rfm_eda.duplicated().sum())

In [ ]:
test = pd.crosstab(df_rfm_eda['order_status'], df_rfm_eda['order_id']).T
test


In [ ]:
test['sum_status']=test.sum(axis=1)

In [ ]:
test.loc[test['sum_status']!=1, :] 

####  Auffälligkeiten
1. Die Nan Werte machen einen sehr geringen Anteil aus. 
2. Da die Zeilen in denen sich die NaN werte befinden, den Order Status delivered haben, werde ich die Daten berücksichten, da der Kauf stattgefunden.



Für die Auswertung von Aufgabe 1. ist der order_status sehr wichtig, da ich nur reale Bestellungen betrachten will.

Deswegen schau ich mir erstmal an wie sich die NaNs zu den relevanten Order Status verhalten

Für die RFM Analyse brauche ich nur die approved, delivered, invoiced, processing und shipped order_status
Daher kann ich canceled, created und unavailable erstmal rausnehmen, da dieser order_status für die Aufgabe nicht relevant ist.

In [ ]:
valid_rfm_status = ['delivered', 'shipped', 'processing', 'invoiced', 'approved']
df_rfm_eda = df_rfm_eda[df_rfm_eda['order_status'].isin(valid_rfm_status)]

In [ ]:
test.loc[test['sum_status']!=1, :].sort_values('sum_status', ascending=False)

## Filterung nach der Bestellung mit den meisten Duplikaten

In [ ]:
order_id = 'ca3625898fbd48669d50701aba51cd5f'

# 1. Filter auf diese Order_ID
order_data = df_rfm_eda[df_rfm_eda['order_id'] == order_id]

order_data.head(63)

In [ ]:
df_rfm_eda.describe()

In [ ]:
con.execute("CREATE TABLE customer_rfm AS SELECT * FROM df_rfm_eda")

### EDA für zweite Kernaufgabe

In [ ]:
# Dataframe für Aufgabe 2. für EDA
df_pc_eda = sql("""
SELECT 
    p.product_id,
    pcnt.product_category_name_english,
    Count(o.order_id) AS order_count,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    oi.price,
    oi.freight_value,
    r.review_score
FROM products p
JOIN order_items oi ON p.product_id = oi.product_id 
JOIN orders o ON oi.order_id = o.order_id
JOIN product_category_name_translation pcnt ON pcnt.product_category_name = p.product_category_name
LEFT JOIN order_reviews r ON o.order_id = r.order_id
WHERE o.order_status IN ('delivered', 'shipped', 'processing', 'invoiced', 'approved')
GROUP BY p.product_id, pcnt.product_category_name_english, o.order_status, o.order_purchase_timestamp,
         o.order_approved_at, oi.price, oi.freight_value, r.review_score
    """)

In [ ]:
df_pc_eda

In [ ]:
df_pc_eda.describe()

In [ ]:
df_pc_eda.dtypes

In [ ]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'product_id': 'category',
              'product_category_name_english': 'category', 
              'order_count': 'Int16', 
              'order_status': 'category', 
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_approved_at': 'datetime64[s]', 
              'price': 'float32',
              'freight_value': 'float32',
              'review_score': 'category',}
df_pc_eda = df_pc_eda.astype(col_dtypes)
df_pc_eda.dtypes

In [ ]:
df_pc_eda.describe()

In [ ]:
df_pc_eda.isna().sum()

In [ ]:
df_pc_eda['order_approved_at'] = df_pc_eda['order_approved_at'].fillna(
    pd.to_datetime(df_pc_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_pc_eda[df_pc_eda.isna().any(axis=1)].head(15)

In [ ]:
print("Gesamte Duplikate:", df_pc_eda.duplicated().sum())

In [ ]:
con.execute("CREATE TABLE product_category AS SELECT * FROM df_pc_eda")

### EDA für dritte Kernaufgabe

In [ ]:
# Dataframe für Aufgabe 2. für EDA
df_service_eda = sql("""
SELECT 
    o.order_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
        r.review_id,
        r.review_score,
        r.review_comment_title,
        r.review_comment_message,
        r.review_creation_date,
        r.review_answer_timestamp,
    oi.product_id,
    pcnt.product_category_name_english,
    Count(oi.product_id) AS product_count,                
    s.seller_id,
    s.seller_city,
    s.seller_state,
    c.customer_city,
    c.customer_state,              
    pcnt.product_category_name_english,
     
FROM orders o
JOIN order_reviews r ON o.order_id = r.order_id
JOIN order_items oi 
        ON o.order_id = oi.order_id
JOIN sellers s
        ON oi.seller_id = s.seller_id
JOIN customers c 
        ON c.customer_id = o.customer_id
JOIN products p 
        ON oi.product_id = p.product_id
JOIN product_category_name_translation pcnt 
        ON pcnt.product_category_name = p.product_category_name
WHERE o.order_status IN ('delivered')
Group BY o.order_id, o.order_status, o.order_purchase_timestamp, o.order_approved_at,
         o.order_delivered_carrier_date, o.order_delivered_customer_date, o.order_estimated_delivery_date,
         r.review_id, r.review_score, r.review_comment_title, r.review_comment_message, 
                     r.review_creation_date, r.review_answer_timestamp,
         oi.product_id, s.seller_id, s.seller_city, s.seller_state, 
                     c.customer_city, c.customer_state, pcnt.product_category_name_english
    """)

In [ ]:
df_service_eda

In [ ]:
df_service_eda.describe()


In [ ]:
df_service_eda.dtypes


In [ ]:
# Anpassung des Datentypes der Spalten, habe das mit Absicht erst jetzt gemacht, da ich mit den NaNs
# den error Paramenter reinmachen muss, aber die Spalte am Ende nicht die gewünschten Datentyp bekommt
col_dtypes = { 
              'order_id': 'category',
              'order_status': 'category',
              'order_approved_at': 'datetime64[s]',
              'order_purchase_timestamp': 'datetime64[s]', 
              'order_delivered_carrier_date': 'datetime64[s]', 
              'order_delivered_customer_date': 'datetime64[s]',
              'order_estimated_delivery_date': 'datetime64[s]',
              'review_id': 'category',
              'review_score': 'Int16',
              'review_comment_title': 'category',
              'review_comment_message': 'category',
              'review_creation_date': 'datetime64[s]',
              'review_answer_timestamp': 'datetime64[s]',
              'product_id': 'category',
              'product_category_name_english': 'category',
              'product_count': 'Int16',
              'seller_id': 'category',
              'seller_city': 'category',
              'seller_state': 'category',
              'customer_city': 'category',
              'customer_state': 'category', 
              
              }
df_service_eda = df_service_eda.astype(col_dtypes)
df_service_eda.dtypes

In [ ]:
df_service_eda.isna().sum()

In [ ]:
df_service_eda['order_approved_at'] = df_service_eda['order_approved_at'].fillna(
    pd.to_datetime(df_service_eda['order_purchase_timestamp']) + pd.Timedelta(seconds=5)
)
df_service_eda[
    df_service_eda[['order_delivered_carrier_date', 'order_delivered_customer_date']].isna().any(axis=1)
].head(15)

In [ ]:
df_service_eda['order_delivered_customer_date'] = df_service_eda['order_delivered_customer_date'].fillna(
    pd.to_datetime(df_service_eda['order_estimated_delivery_date'])
)

df_service_eda['order_delivered_carrier_date'] = df_service_eda['order_delivered_carrier_date'].fillna(
    pd.to_datetime(df_service_eda['order_approved_at']) + pd.Timedelta(days=5)
)
df_service_eda[
    df_service_eda[['order_delivered_carrier_date', 'order_delivered_customer_date']].isna().any(axis=1)
].head(15)

In [ ]:
df_service_eda.describe()

In [ ]:
print("Gesamte Duplikate:", df_service_eda.duplicated().sum())


In [ ]:
order_id = '895ab968e7bb0d5659d16cd74cd1650c'

# 1. Filter auf diese Order_ID
order_data = df_service_eda[(df_service_eda['order_id'] == order_id)]

order_data.head(63).sort_values('order_purchase_timestamp', ascending=True)

In [ ]:
con.execute("CREATE TABLE service_analyse AS SELECT * FROM df_service_eda")


In [ ]:
sql("SHOW TABLES")

In [ ]:
con.close()